In [2]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve

sns.set_theme(style="whitegrid")

file_path = "data/processed/tracts_acs_wind_2024_expanded.parquet"

print("=" * 70)
print("Modeling: Logistic Regression (with Missing Value Handling)")
print("=" * 70)

# ============================================================
# 1. LOAD DATA + DROP MISSING VALUES
# ============================================================

df = pd.read_parquet(file_path)
print(f"Original data: {len(df):,} tracts")

# Select features and target, then drop any rows with missing values
feature_cols = [
    "total_population",
    "total_housing_units",
    "median_household_income",
    "median_home_value",
    "housing_built_2010_or_later",
    "housing_built_before_2000"
]

df_model = df[feature_cols + ["has_wind_farm"]].dropna()
print(f"After dropping missing values: {len(df_model):,} tracts")

X = df_model[feature_cols]
y = df_model["has_wind_farm"]

print(f"\nTarget distribution:\n{y.value_counts(normalize=True).round(3)}")

# ============================================================
# 2. TRAIN / TEST SPLIT + SCALING
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining set: {len(X_train):,}")
print(f"Test set:     {len(X_test):,}")

# ============================================================
# 3. TRAIN LOGISTIC REGRESSION
# ============================================================

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model.fit(X_train_scaled, y_train)
print("\nModel training complete.")

# ============================================================
# 4. EVALUATION
# ============================================================

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("\n" + "=" * 50)
print("MODEL PERFORMANCE")
print("=" * 50)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob):.3f}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["No Wind Farm", "Has Wind Farm"]))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

# ============================================================
# 5. FEATURE IMPORTANCE
# ============================================================

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_[0],
    "Odds Ratio": np.exp(model.coef_[0])
}).sort_values("Coefficient", ascending=False)

print("\n--- Feature Importance ---")
print(coef_df.round(3))

print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Modeling: Logistic Regression (with Missing Value Handling)
Original data: 8,985 tracts
After dropping missing values: 8,518 tracts

Target distribution:
has_wind_farm
0    0.957
1    0.043
Name: proportion, dtype: float64

Training set: 6,388
Test set:     2,130

Model training complete.

MODEL PERFORMANCE

Accuracy: 0.703
ROC-AUC:  0.829

--- Classification Report ---
               precision    recall  f1-score   support

 No Wind Farm       0.99      0.70      0.82      2038
Has Wind Farm       0.11      0.80      0.19        92

     accuracy                           0.70      2130
    macro avg       0.55      0.75      0.50      2130
 weighted avg       0.95      0.70      0.79      2130


--- Confusion Matrix ---
[[1423  615]
 [  18   74]]

--- Feature Importance ---
                       Feature  Coefficient  Odds Ratio
2      median_household_inc

In [3]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

file_path = "data/processed/tracts_acs_wind_2024_expanded.parquet"

print("=" * 70)
print("Improving the Model: Feature Engineering + Logistic Regression")
print("=" * 70)

# ============================================================
# 1. LOAD DATA + CREATE NEW FEATURES
# ============================================================

df = pd.read_parquet(file_path)

# Create derived features
df["pct_recent_housing"] = df["housing_built_2010_or_later"] / (df["total_housing_units"] + 1)
df["housing_density"] = df["total_housing_units"] / (df["total_population"] + 1)
df["income_vs_home_value"] = df["median_household_income"] / (df["median_home_value"] + 1)

# Select final feature set (original + engineered)
feature_cols = [
    "total_population",
    "total_housing_units",
    "median_household_income",
    "median_home_value",
    "housing_built_2010_or_later",
    "housing_built_before_2000",
    "pct_recent_housing",
    "housing_density",
    "income_vs_home_value"
]

# Drop rows with missing values in selected features
df_model = df[feature_cols + ["has_wind_farm"]].dropna()

X = df_model[feature_cols]
y = df_model["has_wind_farm"]

print(f"Final modeling dataset: {len(df_model):,} tracts")
print(f"Features used: {feature_cols}")

# ============================================================
# 2. TRAIN / TEST SPLIT + SCALING
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ============================================================
# 3. TRAIN IMPROVED LOGISTIC REGRESSION
# ============================================================

model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

model.fit(X_train_scaled, y_train)

# ============================================================
# 4. EVALUATE AND COMPARE
# ============================================================

y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print("\n" + "=" * 50)
print("IMPROVED MODEL PERFORMANCE")
print("=" * 50)

print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob):.3f}")

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["No Wind Farm", "Has Wind Farm"]))

print("\n--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

# ============================================================
# 5. NEW FEATURE IMPORTANCE
# ============================================================

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_[0],
    "Odds Ratio": np.exp(model.coef_[0])
}).sort_values("Coefficient", ascending=False)

print("\n--- Improved Feature Importance ---")
print(coef_df.round(3))

print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Improving the Model: Feature Engineering + Logistic Regression
Final modeling dataset: 8,518 tracts
Features used: ['total_population', 'total_housing_units', 'median_household_income', 'median_home_value', 'housing_built_2010_or_later', 'housing_built_before_2000', 'pct_recent_housing', 'housing_density', 'income_vs_home_value']

IMPROVED MODEL PERFORMANCE

Accuracy: 0.737
ROC-AUC:  0.830

--- Classification Report ---
               precision    recall  f1-score   support

 No Wind Farm       0.99      0.73      0.84      2038
Has Wind Farm       0.12      0.80      0.21        92

     accuracy                           0.74      2130
    macro avg       0.55      0.77      0.53      2130
 weighted avg       0.95      0.74      0.81      2130


--- Confusion Matrix ---
[[1495  543]
 [  18   74]]

--- Improved Feature Importance ---
                       

In [2]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

file_path = "data/processed/tracts_acs_wind_2024_expanded.parquet"

print("=" * 70)
print("Comparing Models: Logistic Regression vs HistGradientBoosting")
print("=" * 70)

# ============================================================
# 1. LOAD DATA + RE-CREATE DERIVED FEATURES
# ============================================================

df = pd.read_parquet(file_path)

# Re-create the engineered features
df["pct_recent_housing"] = df["housing_built_2010_or_later"] / (df["total_housing_units"] + 1)
df["housing_density"] = df["total_housing_units"] / (df["total_population"] + 1)
df["income_vs_home_value"] = df["median_household_income"] / (df["median_home_value"] + 1)

feature_cols = [
    "total_population",
    "total_housing_units",
    "median_household_income",
    "median_home_value",
    "housing_built_2010_or_later",
    "housing_built_before_2000",
    "pct_recent_housing",
    "housing_density",
    "income_vs_home_value"
]

df_model = df[feature_cols + ["has_wind_farm"]].copy()

X = df_model[feature_cols]
y = df_model["has_wind_farm"]

print(f"Final dataset size: {len(df_model):,} tracts")

# ============================================================
# 2. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ============================================================
# 3. LOGISTIC REGRESSION (Baseline)
# ============================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
log_model.fit(X_train_scaled, y_train)

y_pred_log = log_model.predict(X_test_scaled)
y_prob_log = log_model.predict_proba(X_test_scaled)[:, 1]

print("\n" + "=" * 50)
print("LOGISTIC REGRESSION (Improved)")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test, y_pred_log):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob_log):.3f}")

# ============================================================
# 4. HISTGRADIENTBOOSTING CLASSIFIER
# ============================================================

hgb_model = HistGradientBoostingClassifier(
    class_weight="balanced",
    max_iter=200,
    learning_rate=0.1,
    random_state=42
)

hgb_model.fit(X_train, y_train)

y_pred_hgb = hgb_model.predict(X_test)
y_prob_hgb = hgb_model.predict_proba(X_test)[:, 1]

print("\n" + "=" * 50)
print("HISTGRADIENTBOOSTING CLASSIFIER")
print("=" * 50)
print(f"Accuracy: {accuracy_score(y_test, y_pred_hgb):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_prob_hgb):.3f}")

print("\n--- Classification Report (HistGradientBoosting) ---")
print(classification_report(y_test, y_pred_hgb, target_names=["No Wind Farm", "Has Wind Farm"]))

print("\n--- Confusion Matrix (HistGradientBoosting) ---")
print(confusion_matrix(y_test, y_pred_hgb))

# ============================================================
# 5. MODEL COMPARISON
# ============================================================

print("\n" + "=" * 50)
print("FINAL MODEL COMPARISON")
print("=" * 50)
print(f"{'Model':<35} {'Accuracy':>10} {'ROC-AUC':>10}")
print("-" * 55)
print(f"{'Logistic Regression (Improved)':<35} {accuracy_score(y_test, y_pred_log):>10.3f} {roc_auc_score(y_test, y_prob_log):>10.3f}")
print(f"{'HistGradientBoostingClassifier':<35} {accuracy_score(y_test, y_pred_hgb):>10.3f} {roc_auc_score(y_test, y_prob_hgb):>10.3f}")
print("=" * 55)

print("\nNote: HistGradientBoosting usually performs better on this type of data.")

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Comparing Models: Logistic Regression vs HistGradientBoosting
Final dataset size: 8,985 tracts


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values